In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [4]:
load_dotenv(override=True)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(pushover_user[0:2])
else:
    print("Pushover user doesn't exist")

if pushover_token:
    print(pushover_token[0:2])
else:
    print("Pushover token doesn't exist")

ue
a3


In [7]:
def push(message):
    print(f"Push this message {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

def record_user_interest(email, name="name is not provided", notes="not provided"):
    push(f"There is an interest from {name} with email {email} and notes {notes}")
    return {"recorded": "OK"}

def record_unknown_question(question):
    push(f"Taking a note of this question {question} that I couldn't answer")
    return {"recorded": "OK"}

push("Hi it is me Ghazwan")

Push this message Hi it is me Ghazwan


In [6]:
user_details_schema = {
    "name": "record_user_details",
    "description": "Use this tool to register user interest in getting in touch",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "User email address"
            },
            "name": {
                "type": "string",
                "description": "User's name"
            },
            "notes": {
                "type": "string",
                "description": "Any additional information about the user and the conversation"
            }
        },
        "required": ["name"],
        "additionalProperties": False
    }
}

unknown_question_schema= {
    "name": "record_unknown_question",
    "description": "recording user's question that LLM couldn't answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "user's question"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [9]:
tools = [
    {"type": "function", "function": record_user_interest},
    {"type": "function", "function": record_unknown_question}
]

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments)
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = OpenAI.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason

        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
            response = OpenAI.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        else:
            done=True
        return response.choices[0].message.content
        